# 📄 Multi-Document PDF RAG Assistant

**Tech Stack:** pypdf · sentence-transformers · FAISS · TinyLlama-1.1B-Chat

Notebook ini menjalankan end-to-end RAG pipeline:
1. Install dependencies
2. Setup project structure
3. Download/upload PDF dataset
4. Indexing pipeline (PDF → chunks → embeddings → FAISS)
5. RAG query (retrieval + generation)
6. Evaluation
7. (Bonus) Launch Streamlit app via ngrok


## 0. Runtime Check
Pastikan GPU aktif: **Settings → Accelerator → GPU T4**

In [2]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

PyTorch version : 2.12.0+cpu
CUDA available  : False


## 1. Imports Setup

Jalankan cell di bawah ini untuk memasukkan direktori `src/` ke dalam `sys.path` dan mengimpor semua modul/file lokal agar siap digunakan untuk testing.

In [3]:
import os
import sys

possible_src_paths = [
    os.path.abspath(os.path.join('..', 'src')),
    os.path.abspath('src')
]

src_path = None
for path in possible_src_paths:
    if os.path.exists(os.path.join(path, 'loader.py')):
        src_path = path
        break

if src_path:
    sys.path.insert(0, src_path)
    PROJECT_ROOT = os.path.dirname(src_path)
    print(f'✅ Folder src ditemukan di: {src_path}')
    print(f'✅ PROJECT_ROOT diatur ke: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = os.path.abspath('.')
    print('❌ ERROR: Folder src tidak ditemukan atau loader.py tidak ada di dalamnya!')
    print(f'   Current Working Directory (os.getcwd()): {os.getcwd()}')
    try:
        print(f'   Isi direktori aktif saat ini: {os.listdir(os.curdir)}')
        if 'src' in os.listdir(os.curdir):
            print(f'   Isi folder src: {os.listdir("src")}')
        elif '../src' in os.listdir(os.curdir) or os.path.exists(os.path.join('..', 'src')):
            print(f'   Isi folder ../src: {os.listdir(os.path.join("..", "src"))}')
    except Exception as e:
        print(f'   Gagal membaca isi direktori: {e}')

if src_path:
    from loader import load_pdfs, get_dataset_summary
    from chunker import chunk_documents
    from embedder import Embedder
    from vectorstore import VectorStore
    from retriever import Retriever
    from rag_pipeline import RAGPipeline
    print('✅ Semua modul lokal dari src berhasil diimpor!')


✅ Folder src ditemukan di: c:\Users\rakad\OneDrive\Dokumen\Projects\nolimit-ds-test-rakadaffa\src
✅ PROJECT_ROOT diatur ke: c:\Users\rakad\OneDrive\Dokumen\Projects\nolimit-ds-test-rakadaffa


c:\Users\rakad\OneDrive\Dokumen\Projects\nolimit-ds-test-rakadaffa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Semua modul lokal dari src berhasil diimpor!


## 2. Dataset Exploration

In [4]:
import sys, json
import os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from loader import get_dataset_summary
import pandas as pd

summary = get_dataset_summary(os.path.join(PROJECT_ROOT, 'data', 'pdfs'))
print(f"Total PDFs  : {summary['total_files']}")
print(f"Total pages : {summary['total_pages']}")
print()
df = pd.DataFrame(summary['files'])
display(df)

Total PDFs  : 30
Total pages : 330



,filename,size_kb,pages,readable
0,aa932e9b0226461291e5f75cc35d2e13c0ab3218.pdf,512.53,9,True
1,aa9edb6b15f25884ea1d47308c271fbf5badd29d.pdf,2173.96,1,True
2,af754bdf786579eb81414d411ef7c19f4e62ace6.pdf,331.32,22,True
3,b243f6218b4f2ca4de2717cf4a2af223b68210db.pdf,18903.85,22,True
4,b47adcf5e54f1d10bf91966ff3439f089610c0f0.pdf,671.85,11,True
5,b4eb68dd1d92f8c56b25e6d52948ba6c8ca8ec83.pdf,3182.43,17,True
6,b705ad3e556b7f152b8ada00da4cb55994262d76.pdf,400.51,6,True
7,b818855b7d3d5822dd578d58a909968f5504497d.pdf,7885.67,32,True
8,c0aceb39c261fc0b324233345b8970b51d1015c9.pdf,295.91,2,True
9,c31721d6245f5689e5d715b1497b2374df7ae4c6.pdf,5036.66,9,True


## 3. Indexing Pipeline

In [5]:
import torch, sys
import os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

from loader import load_pdfs
from chunker import chunk_documents
from embedder import Embedder
from vectorstore import VectorStore
from retriever import Retriever

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Step 1: Load PDFs
documents = load_pdfs(os.path.join(PROJECT_ROOT, 'data', 'pdfs'))

# Step 2: Chunk
chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=100)

# Step 3: Embed
embedder = Embedder(device=DEVICE)
embeddings = embedder.encode_chunks(chunks)

# Step 4: Build FAISS index
vector_store = VectorStore(embedder.embedding_dim)
vector_store.add(embeddings, chunks)
vector_store.save(os.path.join(PROJECT_ROOT, 'faiss_index'))

# Step 5: Build retriever
retriever = Retriever(embedder, vector_store)
print('\n✅ Indexing pipeline complete')

Device: cpu
Found 30 PDF file(s) in 'c:\Users\rakad\OneDrive\Dokumen\Projects\nolimit-ds-test-rakadaffa\data\pdfs'


Loading PDFs: 100%|██████████| 30/30 [00:54<00:00,  1.82s/it]


Extracted text from 329 page(s) across 30 PDF(s)
Generated 2110 chunk(s) from 329 page(s) [size=500, overlap=100]
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1760.40it/s]
c:\Users\rakad\OneDrive\Dokumen\Projects\nolimit-ds-test-rakadaffa\src\embedder.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dim = self.model.get_sentence_embedding_dimension()


Embedding dimension: 384


Batches: 100%|██████████| 33/33 [01:01<00:00,  1.87s/it]


Generated 2110 embeddings of dim 384
VectorStore: 2110 vector(s) indexed
VectorStore saved → c:\Users\rakad\OneDrive\Dokumen\Projects\nolimit-ds-test-rakadaffa\faiss_index

✅ Indexing pipeline complete


## 4. Load TinyLlama LLM

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

LLM_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {LLM_MODEL} on {DEVICE}...')

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
)
if DEVICE == 'cpu':
    llm = llm.to('cpu')

generator = hf_pipeline(
    'text-generation',
    model=llm,
    tokenizer=tokenizer,
    device=0 if DEVICE == 'cuda' else -1,
)
print('✅ LLM loaded')

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cpu...


c:\Users\rakad\OneDrive\Dokumen\Projects\nolimit-ds-test-rakadaffa\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rakad\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:08<00:00, 23.83

✅ LLM loaded


## 5. RAG Query Function

In [7]:
FALLBACK = 'I could not find relevant information in the document collection.'

PROMPT_TEMPLATE = """<|system|>
You are a helpful assistant. Answer questions strictly using the provided context.
If the answer is not in the context, say: \"{fallback}\"
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>
"""

def rag_query(question, top_k=5):
    results  = retriever.retrieve(question, top_k=top_k)
    context  = retriever.format_context(results)
    citations = retriever.format_citations(results)

    prompt = PROMPT_TEMPLATE.format(
        fallback=FALLBACK, context=context, question=question
    )

    output = generator(
        prompt,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    full_text = output[0]['generated_text']
    answer    = full_text[len(prompt):].strip()

    return {
        'question': question,
        'answer':   answer or FALLBACK,
        'sources':  citations,
    }

print('✅ rag_query() ready')

✅ rag_query() ready


## 6. Run a Test Query

In [ ]:
QUESTION = 'What is the main topic discussed in the documents?'

result = rag_query(QUESTION, top_k=5)

print('=' * 60)
print(f'QUESTION: {result["question"]}')
print('=' * 60)
print(f'ANSWER:\n{result["answer"]}')
print()
print('SOURCES:')
for src in result['sources']:
    print(f"  [{src['source_num']}] {src['filename']} — page {src['page_number']} (score={src['score']})")
    print(f"      {src['snippet'][:150]}...")
    print()

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


QUESTION: What is the main topic discussed in the documents?
ANSWER:
The main topic discussed in the documents listed in the text material is the framework for managing humanitarian data within OCHA. These documents outline the guidelines and policies that govern the management of data at OCHA, including the use of technology standards, the allocation of resources, and the governance of data.

SOURCES:
  [1] f997183fa6bf1ef75e31406d98f025c16e527d20.pdf — page 44 (score=0.4616)
      or the Coordination of Humanitarian Aﬀairs (OCHA), Policy Instruction 
on Technology Standards, https://unitednations.sharepoint.com/sites/OCHAHub/
IM...

  [2] f997183fa6bf1ef75e31406d98f025c16e527d20.pdf — page 44 (score=0.4604)
      945. Charter of the United Nations: 
https://www.un.org/en/about-us/un-charter. 
UN General Assembly, 1948. Universal Declaration of Human Rights: 
ht...

  [3] b47adcf5e54f1d10bf91966ff3439f089610c0f0.pdf — page 7 (score=0.424)
      nt and implementation of the model.
What

## 7. Evaluation

In [10]:
import pandas as pd

EVAL_QUESTIONS = [
    'What is the main topic discussed in the first document?',
    'Summarize the key points of the document collection.',
    'What specific information is provided about any technical topic?',
    'What conclusions or recommendations are made?',
    'Are there any data, statistics, or numbers mentioned?',
]

rows = []
for q in EVAL_QUESTIONS:
    r = rag_query(q, top_k=5)
    top_src = r['sources'][0] if r['sources'] else {}
    rows.append({
        'question':          q,
        'answer_preview':    r['answer'][:200],
        'top_source':        top_src.get('filename', '-'),
        'top_page':          top_src.get('page_number', '-'),
        'top_score':         top_src.get('score', 0),
        'has_fallback':      r['answer'] == 'I could not find relevant information in the document collection.',
    })

df_eval = pd.DataFrame(rows)
display(df_eval)

# Save results
df_eval.to_markdown(os.path.join(PROJECT_ROOT, 'outputs', 'evaluation_results.md'), index=False)
print('\n✅ Evaluation saved → outputs/evaluation_results.md')

[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

,question,answer_preview,top_source,top_page,top_score,has_fallback
0,What is the main topic discussed in the first ...,The main topic discussed in the first document...,af754bdf786579eb81414d411ef7c19f4e62ace6.pdf,4,0.4048,False
1,Summarize the key points of the document colle...,The document collection includes various secti...,b4eb68dd1d92f8c56b25e6d52948ba6c8ca8ec83.pdf,2,0.3681,False
2,What specific information is provided about an...,The context provided in the given text materia...,f997183fa6bf1ef75e31406d98f025c16e527d20.pdf,44,0.4819,False
3,What conclusions or recommendations are made?,"The given text contains multiple sources, incl...",b47adcf5e54f1d10bf91966ff3439f089610c0f0.pdf,10,0.4864,False
4,"Are there any data, statistics, or numbers men...","Yes, there are several data, statistics, and n...",b4eb68dd1d92f8c56b25e6d52948ba6c8ca8ec83.pdf,10,0.5032,False


OSError: Cannot save file into a non-existent directory: 'c:\Users\rakad\OneDrive\Dokumen\Projects\nolimit-ds-test-rakadaffa\outputs'

## 8. Save Sample Results

In [ ]:
import json, datetime

sample_result = rag_query(QUESTION, top_k=5)

md_content = f"""# Sample RAG Results

Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## Question
{sample_result['question']}

## Answer
{sample_result['answer']}

## Sources
"""

for src in sample_result['sources']:
    md_content += f"""\n### [{src['source_num']}] {src['filename']} — Page {src['page_number']}
- **Similarity Score:** {src['score']}
- **Snippet:** {src['snippet']}
"""

with open(os.path.join(PROJECT_ROOT, 'outputs', 'sample_results.md'), 'w') as f:
    f.write(md_content)

print('✅ Sample results saved → outputs/sample_results.md')
print(md_content[:500])

## 13. Launch Streamlit App Locally (Untested)

Untuk menjalankan UI chatbot secara lokal, cara terbaik adalah menjalankannya dari terminal/powershell Anda:
```bash
streamlit run src/streamlit_app.py
```

Atau, Anda dapat menjalankan cell di bawah ini untuk memulai server Streamlit lokal secara otomatis dari notebook ini.

In [ ]:
import subprocess
import webbrowser
import time
import os

streamlit_path = os.path.join(PROJECT_ROOT, 'src', 'streamlit_app.py')

print('Memulai Streamlit server lokal...')
try:
    proc = subprocess.Popen(
        ['streamlit', 'run', streamlit_path],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    
    time.sleep(3)
    
    webbrowser.open('http://localhost:8501')
    print('✅ Streamlit App berhasil dijalankan!')
    print('🌐 Silakan akses di browser Anda: http://localhost:8501')
    print('💡 Untuk mematikan server Streamlit, silakan Restart Kernel notebook ini.')
except Exception as e:
    print(f'❌ Gagal menjalankan Streamlit: {e}')
    print('Cobalah untuk menjalankannya secara manual dari terminal dengan perintah: streamlit run src/streamlit_app.py')
